<a href="https://colab.research.google.com/github/raushan95a/Waste-segregation/blob/main/Waste_Detection(executed).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Google Drive folder ID for the dataset
GOOGLE_DRIVE_FOLDER_ID = "1VJE5qj9DjZ9rZy6JT_-AIWVTwgbnRLDl"

# Detect environment and set up paths
IS_COLAB = False
DATASET_ROOT = None

try:
    from google.colab import drive
    IS_COLAB = True
    print("✓ Google Colab detected - mounting Google Drive...")
    drive.mount('/content/drive')

    # First try the default path
    default_path = '/content/drive/My Drive/Waste_Dataset'
    if os.path.exists(default_path):
        DATASET_ROOT = default_path
        print(f"✓ Found dataset at: {DATASET_ROOT}")
    else:
        # Try to download from the provided Google Drive folder
        print(f"Downloading dataset from Google Drive (ID: {GOOGLE_DRIVE_FOLDER_ID})...")
        try:
            import subprocess
            subprocess.run(["pip", "install", "gdown", "-q"], check=True)
            import gdown

            # Download the folder
            os.makedirs('/content/drive/My Drive/Waste_Dataset', exist_ok=True)
            gdown.download_folder(
                f"https://drive.google.com/drive/folders/{GOOGLE_DRIVE_FOLDER_ID}?usp=drive_link",
                output='/content/drive/My Drive/Waste_Dataset',
                quiet=False,
                use_cookies=False
            )
            DATASET_ROOT = '/content/drive/My Drive/Waste_Dataset'
            print(f"✓ Dataset downloaded to: {DATASET_ROOT}")
        except Exception as e:
            print(f"✗ Download failed: {e}")
            print("Trying alternative download method...")
            DATASET_ROOT = '/content/drive/My Drive/Waste_Dataset'

    os.chdir('/content/drive/My Drive')

except ImportError:
    print("✓ Local environment detected")
    # For local execution, construct path relative to current location
    DATASET_ROOT = os.path.abspath('Waste_Dataset')

# Verify dataset exists
if not os.path.exists(DATASET_ROOT):
    print(f"\n⚠ Dataset not found at: {DATASET_ROOT}")
    print(f"  Colab users: Make sure the Google Drive folder is accessible")
    print(f"  Local users: Ensure Waste_Dataset folder exists at: {DATASET_ROOT}")
else:
    print(f"✓ Dataset found at: {DATASET_ROOT}")
    # List contents
    if os.path.exists(os.path.join(DATASET_ROOT, 'Images_merged')):
        images = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'Images_merged')) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"  Images: {images} files")
    if os.path.exists(os.path.join(DATASET_ROOT, 'Annotations_merged')):
        annos = len([f for f in os.listdir(os.path.join(DATASET_ROOT, 'Annotations_merged')) if f.endswith('.xml')])
        print(f"  Annotations: {annos} files")

print(f"\nEnvironment: {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset root: {DATASET_ROOT}")

✓ Google Colab detected - mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Found dataset at: /content/drive/My Drive/Waste_Dataset
✓ Dataset found at: /content/drive/My Drive/Waste_Dataset
  Images: 785 files
  Annotations: 785 files

Environment: Google Colab
Dataset root: /content/drive/My Drive/Waste_Dataset


In [2]:
## Step 2: Install Dependencies and Clone Monk

import subprocess

# Clone Monk if not already present
monk_path = "Monk_Object_Detection"
if not os.path.exists(monk_path):
    print("Cloning Monk Object Detection...")
    try:
        subprocess.run(["git", "clone",
                       "https://github.com/Tessellate-Imaging/Monk_Object_Detection.git"],
                      check=True)
        print("✓ Repository cloned")
    except Exception as e:
        print(f"Warning: {e}")
else:
    print(f"✓ Monk repository exists: {monk_path}")

# Install dependencies
def install_package(pkg):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        return True
    except:
        return False

packages = ['xmltodict', 'pycocotools', 'tqdm', 'opencv-python', 'numpy', 'pandas']
for pkg in packages:
    if install_package(pkg):
        print(f"✓ {pkg}")

print("✓ All dependencies ready")

✓ Monk repository exists: Monk_Object_Detection
✓ xmltodict
✓ pycocotools
✓ tqdm
✓ opencv-python
✓ numpy
✓ pandas
✓ All dependencies ready


In [3]:
# Skip - requirements already handled above
print("✓ Installation already completed")

✓ Installation already completed


In [4]:
# Working directory setup
print(f"Current working directory: {os.getcwd()}")
print(f"Dataset root: {DATASET_ROOT}")

Current working directory: /content/drive/My Drive
Dataset root: /content/drive/My Drive/Waste_Dataset


In [5]:
# xmltodict already installed
print("✓ xmltodict ready")

✓ xmltodict ready


In [6]:
import os
import sys
import numpy as np
import pandas as pd

import xmltodict
import json
from tqdm.notebook import tqdm

from pycocotools.coco import COCO

In [7]:
root_dir = "Waste_Dataset/";
img_dir = "Images_merged/";
anno_dir = "Annotations_merged/";

In [8]:
 files = os.listdir(root_dir + anno_dir);
 print(files)

['garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.xml', 'istockphoto-893136716-640x640_jpg.rf.b806a96cae0228abb8f1e1a5c78655ab.xml', 'WhatsApp-Image-2022-01-23-at-4-46-18-PM_jpeg.rf.26497050a3571d4d0737c3861cc1bfac.xml', 'WhatsApp-Image-2022-06-18-at-8-51-15-PM--1-_jpeg.rf.b6c0b957b69b409209df9a86de2e04b3.xml', 'garbage-district-central-sep-12-2017-athar-khan-1505665663_jpg.rf.b1b53775634a826f4e71e9c72caa6cb3.xml', '5d6841fbd44e8_jpg.rf.88bede017c5c70a90e101061ebf8b92f.xml', 'download_jpg.rf.23c68253ada58f61bf9bbc8ee5f318ce.xml', 'WhatsApp-Image-2022-06-18-at-8-51-11-PM_jpeg.rf.4c8c5a8f28035d4b54e040f491605d9b.xml', 'WhatsApp-Image-2022-05-18-at-6-25-26-PM_jpeg.rf.8f8c12d1c06a7a3523548e44d19322ec.xml', 'istockphoto-1326547050-640x640_jpg.rf.03070d017a0bb043dedd331861d803d7.xml', 'dc-Cover-nepno9mm6htq9bkrchikfdm5l6-20160509001827-Medi_jpg.rf.86ab5d02a92a53192ef205c46199fc6d.xml', 'ezgif-frame-007_jpg.rf.377c59bec53e86edce1e3f7c16cfb695.xml', 'la-tk-20160420-003_jpg.rf.cb0b1af1b135a3938

In [9]:
combined = [];
for i in tqdm(range(len(files))):
    annoFile = root_dir + "/" + anno_dir + "/" + files[i];
    f = open(annoFile, 'r');
    my_xml = f.read();
    anno = dict(dict(xmltodict.parse(my_xml))["annotation"])
    fname = anno["filename"];
    label_str = "";
    if(type(anno["object"]) == list ):
        for j in range(len(anno["object"])):
            obj = dict(anno["object"][j]);
            label = anno["object"][j]["name"];
            bbox = dict(anno["object"][j]["bndbox"])
            x1 = bbox["xmin"];
            y1 = bbox["ymin"];
            x2 = bbox["xmax"];
            y2 = bbox["ymax"];
            if(j == len(anno["object"])-1):
                label_str += x1 + " " + y1 + " " + x2 + " " + y2 + " " + label;
            else:
                label_str += x1 + " " + y1 + " " + x2 + " " + y2 + " " + label + " ";
    else:
        obj = dict(anno["object"]);
        label = anno["object"]["name"];
        bbox = dict(anno["object"]["bndbox"])
        x1 = bbox["xmin"];
        y1 = bbox["ymin"];
        x2 = bbox["xmax"];
        y2 = bbox["ymax"];

        label_str += x1 + " " + y1 + " " + x2 + " " + y2 + " " + label;


    combined.append([fname, label_str])

  0%|          | 0/785 [00:00<?, ?it/s]

In [10]:
print(combined)

[['garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg', '72 91 416 416 Garbage'], ['istockphoto-893136716-640x640_jpg.rf.b806a96cae0228abb8f1e1a5c78655ab.jpg', '127 1 416 404 Garbage'], ['WhatsApp-Image-2022-01-23-at-4-46-18-PM_jpeg.rf.26497050a3571d4d0737c3861cc1bfac.jpg', '4 127 417 313 Garbage'], ['WhatsApp-Image-2022-06-18-at-8-51-15-PM--1-_jpeg.rf.b6c0b957b69b409209df9a86de2e04b3.jpg', '32 59 412 359 Garbage'], ['garbage-district-central-sep-12-2017-athar-khan-1505665663_jpg.rf.b1b53775634a826f4e71e9c72caa6cb3.jpg', '1 177 417 416 Garbage'], ['5d6841fbd44e8_jpg.rf.88bede017c5c70a90e101061ebf8b92f.jpg', '1 78 350 405 Garbage'], ['download_jpg.rf.23c68253ada58f61bf9bbc8ee5f318ce.jpg', '1 1 416 343 Garbage'], ['WhatsApp-Image-2022-06-18-at-8-51-11-PM_jpeg.rf.4c8c5a8f28035d4b54e040f491605d9b.jpg', '1 1 416 315 Garbage 170 275 320 416 Garbage'], ['WhatsApp-Image-2022-05-18-at-6-25-26-PM_jpeg.rf.8f8c12d1c06a7a3523548e44d19322ec.jpg', '55 163 401 292 Garbage'], ['istockphoto-1326547050-

In [11]:
df = pd.DataFrame(combined, columns = ['ID', 'Label']);
df.to_csv(root_dir + "/train_labels.csv", index=False);

In [12]:
import os
import numpy as np
import cv2
import dicttoxml
import xml.etree.ElementTree as ET
from xml.dom.minidom import parseString
from tqdm import tqdm
import shutil
import json
import pandas as pd

In [13]:
root = "Waste_Dataset/";
img_dir = "Images_merged/";
anno_file = "train_labels.csv";

In [14]:
dataset_path = root;
images_folder = root + "/" + img_dir;
annotations_path = root + "/annotations/";

In [15]:
if not os.path.isdir(annotations_path):
    os.mkdir(annotations_path)

input_images_folder = images_folder;
input_annotations_path = root + "/" + anno_file;

In [16]:
output_dataset_path = root;
output_image_folder = input_images_folder;
output_annotation_folder = annotations_path;

tmp = img_dir.replace("/", "");
output_annotation_file = output_annotation_folder + "/instances_" + tmp + ".json";
output_classes_file = output_annotation_folder + "/classes.txt";

In [17]:
if not os.path.isdir(output_annotation_folder):
    os.mkdir(output_annotation_folder);

In [18]:
df = pd.read_csv(input_annotations_path);
columns = df.columns

In [19]:
delimiter = " ";

In [20]:
list_dict = [];
anno = [];
for i in range(len(df)):
    img_name = df[columns[0]][i];
    labels = df[columns[1]][i];
    tmp = labels.split(delimiter);
    for j in range(len(tmp)//5):
        label = tmp[j*5+4];
        if(label not in anno):
            anno.append(label);
    anno = sorted(anno)

for i in tqdm(range(len(anno))):
    tmp = {};
    tmp["supercategory"] = "master";
    tmp["id"] = i;
    tmp["name"] = anno[i];
    list_dict.append(tmp);

anno_f = open(output_classes_file, 'w');
for i in range(len(anno)):
    anno_f.write(anno[i] + "\n");
anno_f.close();

100%|██████████| 1/1 [00:00<00:00, 8507.72it/s]


In [21]:
coco_data = {};
coco_data["type"] = "instances";
coco_data["images"] = [];
coco_data["annotations"] = [];
coco_data["categories"] = list_dict;
image_id = 0;
annotation_id = 0;


for i in tqdm(range(len(df))):
    img_name = df[columns[0]][i];
    labels = df[columns[1]][i];
    tmp = labels.split(delimiter);
    image_in_path = input_images_folder + "/" + img_name;
    print(image_in_path)
    img = cv2.imread(image_in_path, 1);
    h, w, c = img.shape;

    images_tmp = {};
    images_tmp["file_name"] = img_name;
    images_tmp["height"] = h;
    images_tmp["width"] = w;
    images_tmp["id"] = image_id;
    coco_data["images"].append(images_tmp);


    for j in range(len(tmp)//5):
        x1 = int(tmp[j*5+0]);
        y1 = int(tmp[j*5+1]);
        x2 = int(tmp[j*5+2]);
        y2 = int(tmp[j*5+3]);
        label = tmp[j*5+4];
        annotations_tmp = {};
        annotations_tmp["id"] = annotation_id;
        annotation_id += 1;
        annotations_tmp["image_id"] = image_id;
        annotations_tmp["segmentation"] = [];
        annotations_tmp["ignore"] = 0;
        annotations_tmp["area"] = (x2-x1)*(y2-y1);
        annotations_tmp["iscrowd"] = 0;
        annotations_tmp["bbox"] = [x1, y1, x2-x1, y2-y1];
        annotations_tmp["category_id"] = anno.index(label);

        coco_data["annotations"].append(annotations_tmp)
    image_id += 1;

outfile =  open(output_annotation_file, 'w');
json_str = json.dumps(coco_data, indent=4);
outfile.write(json_str);
outfile.close();

  2%|▏         | 15/785 [00:00<00:05, 142.37it/s]

Waste_Dataset//Images_merged//garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg
Waste_Dataset//Images_merged//istockphoto-893136716-640x640_jpg.rf.b806a96cae0228abb8f1e1a5c78655ab.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-4-46-18-PM_jpeg.rf.26497050a3571d4d0737c3861cc1bfac.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-51-15-PM--1-_jpeg.rf.b6c0b957b69b409209df9a86de2e04b3.jpg
Waste_Dataset//Images_merged//garbage-district-central-sep-12-2017-athar-khan-1505665663_jpg.rf.b1b53775634a826f4e71e9c72caa6cb3.jpg
Waste_Dataset//Images_merged//5d6841fbd44e8_jpg.rf.88bede017c5c70a90e101061ebf8b92f.jpg
Waste_Dataset//Images_merged//download_jpg.rf.23c68253ada58f61bf9bbc8ee5f318ce.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-51-11-PM_jpeg.rf.4c8c5a8f28035d4b54e040f491605d9b.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-05-18-at-6-25-26-PM_jpeg.rf.8f8c12d1c06a7a3523548e44d19322ec.jpg
Waste_Dataset//Images_merged//istockphoto-1326

  6%|▌         | 46/785 [00:00<00:05, 143.03it/s]

Waste_Dataset//Images_merged//ezgif-frame-022_jpg.rf.5d94c37dcdba82e59edd1bbf8f5e908b.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-50-08-PM_jpeg.rf.117caf9e54f46b4528c57ff4074466cc.jpg
Waste_Dataset//Images_merged//ezgif-frame-011_jpg.rf.ba51740e271096ae4c53285dbc5a1348.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-29-03-PM--1-_jpeg.rf.ef51d3e6be3228566b6eb030e04a2591.jpg
Waste_Dataset//Images_merged//1000_F_210350580_GFGKcLMzeOvWfdnNamPEU8NnolHqKwlQ_jpg.rf.bce5a6656182864c6c0a946b80bb4d91.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-05-18-at-6-24-57-PM--5-_jpeg.rf.4759bb1c11351921098b2488ef3f7937.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-05-18-at-6-24-57-PM--4-_jpeg.rf.77ad19488c74350cced8810cc553d08d.jpg
Waste_Dataset//Images_merged//Garbage--4-1613817183-3_jpg.rf.7117e90936b468efd8981ae3d5abbd9e.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-51-06-PM_jpeg.rf.fe627ac3cab22774d1de961b0ea44eb2.jpg
Waste_Datas

 10%|▉         | 75/785 [00:00<00:05, 137.13it/s]

...

Waste_Dataset//Images_merged//ezgif-frame-033_jpg.rf.9981694c72c2c9a11db99676d2ccec3d.jpg
Waste_Dataset//Images_merged//FTVUrUpUsAAbQTh_1200x768_jpg.rf.46fbb54756f60aee82126b892d9b9441.jpg
Waste_Dataset//Images_merged//istockphoto-845816364-612x612_jpg.rf.5f141e329a2df9dfa71115c07f349727.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-28-14-PM_jpeg.rf.67ae309165d7963a839c216773d12680.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-29-06-PM_jpeg.rf.5aa4fdcb89db22b40701e222ed3230b8.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-28-54-PM_jpeg.rf.c1eb5a42b38f13ad540be8fd43b7d421.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-24-36-PM_jpeg.rf.b5d448c9ea941ebd738dfe5e7588ae0e.jpg
Waste_Dataset//Images_merged//960x0_jpg.rf.885430040a7f093abbfb00e042fa83a1.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-05-18-at-6-24-57-PM--9-_jpeg.rf.fe818ccc8659c276da34427dfd39632a.jpg
Waste_Dataset//Images_merged//ezgif-frame-011_jp

 96%|█████████▌| 751/785 [00:10<00:00, 128.48it/s]

Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-4-45-34-PM_jpeg.rf.500aa766527a930b1070aab5d45768a9.jpg
Waste_Dataset//Images_merged//ezgif-frame-003_jpg.rf.e82d7a25524a3022f3d56de2873aced0.jpg
Waste_Dataset//Images_merged//garbage-everywhere-municipal-waste-heap-where-every-day-cca-tons-dumped-148298207_jpg.rf.035db187bd0f1c7d50c3b5378555fae4.jpg
Waste_Dataset//Images_merged//Atlantic-Garbage-Patch-3-537x420_jpg.rf.2199e70e46ca61d1627a5d973b381eca.jpg
Waste_Dataset//Images_merged//pexels-photo-938044_jpg.rf.7dc41b4559d55fedf4db0b1806504f93.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-21-25-PM_jpeg.rf.1020680d8167ebe26bcb0ae6a7af221d.jpg
Waste_Dataset//Images_merged//pexels-photo-2768961_jpeg.rf.3c3c162bde5abb88a184d32c4b3e6de0.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-01-23-at-5-25-47-PM_jpeg.rf.38bfc2a9ce5018ec85b77ee9fe7050c5.jpg
Waste_Dataset//Images_merged//ezgif-frame-009_jpg.rf.6c3309a465a32393180e875bef9fe8a1.jpg
Waste_Dataset//Images_

100%|██████████| 785/785 [00:11<00:00, 71.22it/s] 

Waste_Dataset//Images_merged//how-much-garbage-does-average-person-produce2-1636129399206_jpg.rf.400d24806bb453f035b461d638bff36a.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-51-12-PM_jpeg.rf.52030a85e801871bb13534b24cb92707.jpg
Waste_Dataset//Images_merged//year-ender-2018-landfill-india-660x330_jpg.rf.6173cc47b7e8c6a4a48752d9e3a74636.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-06-18-at-8-50-12-PM_jpeg.rf.236173285f14668483fdaddf653150e7.jpg
Waste_Dataset//Images_merged//WhatsApp-Image-2022-05-18-at-6-24-57-PM--17-_jpeg.rf.55dc5f92a4efe1430c020355f557272e.jpg
Waste_Dataset//Images_merged//801533-garbage-29_jpg.rf.ad48ea0a78139bde340f5329e0c1d5f1.jpg
Waste_Dataset//Images_merged//Garbage--6-1613817201-5_jpg.rf.be6dc56964793773405f45ac0dfb29aa.jpg
Waste_Dataset//Images_merged//pexels-photo-938044_jpg.rf.8eedcfa71d02716b7fb17170153fbd44.jpg
Waste_Dataset//Images_merged//ezgif-frame-018_jpg.rf.1339386d912c3873bbd1c79c9226527a.jpg
Waste_Dataset//Images_merged/

In [22]:
import os
import sys
print(sys.path.append("Monk_Object_Detection/5_pytorch_retinanet/lib/"))

None


In [23]:
import sys
sys.path.append('Monk_Object_Detection/5_pytorch_retinanet/lib/')

!sed -i '/assert torch.__version__.split/s/^/# Original: & # Commented out by Colab AI to avoid version conflict/' Monk_Object_Detection/5_pytorch_retinanet/lib/train_detector.py
!sed -i -E 's/(, )?verbose=True//g' Monk_Object_Detection/5_pytorch_retinanet/lib/train_detector.py
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q

from train_detector import Detector
gtf = Detector();

In [24]:
root_dir = "./";
coco_dir="Waste_Dataset";
img_dir = "./";
set_dir = "Images_merged";

In [25]:
gtf.Train_Dataset(root_dir,coco_dir, img_dir, set_dir, batch_size=2, use_gpu=True)

loading annotations into memory...
Done (t=0.02s)
creating index...
index created!
Num training images: 785


In [26]:
gtf.system_dict["local"]["dataset_train"].classes

{'Garbage': 0}

In [27]:
gtf.Model(model_name="resnet50");

In [28]:
gtf.Set_Hyperparams(lr=0.0001, print_interval=20)

In [29]:
gtf.Train(num_epochs=8, output_model_name="final_model.pt");

Epoch: 0 | Iteration: 0 | Classification loss: 1.13086 | Regression loss: 0.92618 | Running loss: 2.05704
Epoch: 0 | Iteration: 20 | Classification loss: 0.27918 | Regression loss: 0.84094 | Running loss: 1.78974
Epoch: 0 | Iteration: 40 | Classification loss: 0.70590 | Regression loss: 0.82875 | Running loss: 1.52656
Epoch: 0 | Iteration: 60 | Classification loss: 0.92859 | Regression loss: 1.07852 | Running loss: 1.39092
Epoch: 0 | Iteration: 80 | Classification loss: 0.40566 | Regression loss: 0.72238 | Running loss: 1.33159
Epoch: 0 | Iteration: 100 | Classification loss: 0.75475 | Regression loss: 0.99184 | Running loss: 1.33774
Epoch: 0 | Iteration: 120 | Classification loss: 0.24458 | Regression loss: 0.55584 | Running loss: 1.29730
Epoch: 0 | Iteration: 140 | Classification loss: 0.64846 | Regression loss: 1.20556 | Running loss: 1.25657
Epoch: 0 | Iteration: 160 | Classification loss: 0.41351 | Regression loss: 0.61512 | Running loss: 1.23304
Epoch: 0 | Iteration: 180 | Classi

In [37]:
import os
import sys
sys.path.append("Monk_Object_Detection/5_pytorch_retinanet/lib/");

# Fix UnpicklingError (1): Set weights_only=False for torch.load
!sed -i "s/torch.load(model_path)/torch.load(model_path, weights_only=False)/g" Monk_Object_Detection/5_pytorch_retinanet/lib/infer_detector.py

# Fix UnpicklingError (2): Add DataParallel to safe globals for unpickling
# Insert import and add_safe_globals before the class definition
!sed -i '/class Infer:/i\import torch.serialization\ntorch.serialization.add_safe_globals([torch.nn.parallel.data_parallel.DataParallel])' Monk_Object_Detection/5_pytorch_retinanet/lib/infer_detector.py

In [35]:
import sys
# Ensure the module is reloaded after modification
if 'infer_detector' in sys.modules:
    del sys.modules['infer_detector']
from infer_detector import Infer

In [38]:
gtf = Infer();

In [39]:
print(gtf.Model(model_path="final_model.pt"))

None


In [40]:
f = open("Waste_Dataset/annotations/classes.txt", 'r');
class_list = f.readlines();
f.close();
for i in range(len(class_list)):
    class_list[i] = class_list[i][:-1]

In [41]:
len(class_list)

1

In [44]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [47]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [49]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [51]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [53]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [55]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [57]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [59]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [61]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.


In [63]:
img_path = "Waste_Dataset/Images_merged/garbage_jpg.rf.35f301858bf25a189e0bda2bf511fd79.jpg";
result = gtf.Predict(img_path, class_list, vis_threshold=0.4);

# Check if result is None before unpacking
if result is not None:
    scores, labels, boxes = result
    from IPython.display import Image
    Image(filename='output.jpg')
else:
    print("No objects detected for this image.")

No Boxes detected
No objects detected for this image.
